# TP07: convolutional neural networks

**MSA-DATB09-03 · Neural Networks & Deep Learning Foundations · Session 7 / 8** · Guillaume Saint-Cirgue

*Estimated time: 85 minutes*

In the course we computed convolutions, output shapes, pooling and receptive fields by hand, and we saw why a
convolutional network beats a dense one on images with far fewer parameters. In this notebook you first code
the convolution yourself with NumPy loops (and check it against PyTorch), then you predict shapes and
parameter counts, then you train a small CNN against the dense network of session 5 on FashionMNIST. You
finish with transfer learning: a ResNet-18 pretrained on ImageNet, adapted to CIFAR-10 in a few minutes.

**How it works.** Each exercise is a cell with a zone to complete:

```python
### START CODE HERE ###

### END CODE HERE ###
```

Write your code between the two markers, then run the cell: the `assert` lines below the zone check your
answer and print `Exercise validated!` when it is correct. If an assert fails, read its message: it tells
you what to look at.

**Run it** on Google Colab (File > Upload notebook) or locally with the course `requirements.txt`. Parts A
to C run on a CPU in a few minutes. Part D (ResNet-18) also runs on a CPU with small images, but a Colab GPU
(Runtime > Change runtime type > GPU) makes it much faster.

## Imports

- `numpy`: the convolution by hand in Part A.
- `matplotlib`: images and feature maps.
- `torch`, `torch.nn`: networks; `torch.nn.functional as F` gives the functions behind the layers
  (`F.conv2d`, `F.max_pool2d`, `F.interpolate`, ...).
- `torchvision.datasets`: FashionMNIST and CIFAR-10; `torchvision.models`: the pretrained ResNet-18.

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights

FAST_RUN = os.environ.get("FAST_RUN", "0") == "1"   # True: smaller datasets and fewer epochs for a quick check
DATA_DIR = os.environ.get("DATA_DIR", "./data")
np.random.seed(0)
torch.manual_seed(0)
torch.set_num_threads(4)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("FAST_RUN:", FAST_RUN, "| device:", device)

# Part A: convolution and pooling by hand

### Exercise 1: a convolution with loops

Write `conv2d_valid(X, K)` for a 2-D image `X` of shape `(n, n)` and a kernel `K` of shape `(k, k)`, without
padding and with stride 1. The output has shape `(n - k + 1, n - k + 1)` and

$$Y_{i,j} = \sum_{u=0}^{k-1}\sum_{v=0}^{k-1} K_{u,v}\, X_{i+u,\,j+v}$$

In NumPy, `X[i:i+k, j:j+k]` is the patch under the kernel and `np.sum(patch * K)` the sum of the element-wise
products.

The asserts check Exercise 2 of the course, then compare with PyTorch's `F.conv2d(input, weight)`. `F.conv2d`
wants 4-D tensors: `input` of shape `(N, C_in, H, W)` (a batch of N images with C_in channels) and `weight` of
shape `(C_out, C_in, k, k)`; that is why we use `.view(1, 1, n, n)`.

> Hint: two nested `for` loops over `i` and `j`, and `Y = np.zeros((n - k + 1, n - k + 1))` first.

In [ ]:
def conv2d_valid(X, K):
    n, k = X.shape[0], K.shape[0]
    ### START CODE HERE ###
    Y = None
    ### END CODE HERE ###
    return Y


X_ex = np.array([[1, 1, 1, 4, 4], [1, 2, 1, 4, 4], [1, 1, 1, 4, 4], [1, 1, 2, 4, 4], [1, 1, 1, 4, 4]], dtype=float)
K_ex = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=float)
Y_ex = conv2d_valid(X_ex, K_ex)
assert Y_ex is not None and Y_ex.shape == (3, 3), "the output of a 3x3 kernel on a 5x5 image is 3x3"
assert np.allclose(Y_ex, [[0, -8, -9], [-1, -8, -8], [-1, -9, -8]]), f"course Exercise 2 expects another result:\n{Y_ex}"
_X, _K = np.random.randn(6, 6), np.random.randn(3, 3)
_ref = F.conv2d(torch.tensor(_X).view(1, 1, 6, 6), torch.tensor(_K).view(1, 1, 3, 3)).numpy()[0, 0]
assert np.allclose(conv2d_valid(_X, _K), _ref), "your convolution differs from F.conv2d"
print("Exercise validated!")
print(Y_ex)

### Exercise 2: the output size formula

Write `output_size(n, k, p=0, s=1)`: the size of the output of a convolution (or a pooling) with kernel size
`k`, padding `p` and stride `s` on an input of size `n`:

$$n_{\text{out}} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1$$

> Hint: in Python, `a // b` is the integer division (the floor for positive numbers).

In [ ]:
def output_size(n, k, p=0, s=1):
    ### START CODE HERE ###
    out = None
    ### END CODE HERE ###
    return out


assert output_size(5, 3) == 3, "5x5 image, 3x3 kernel, no padding: 3"
assert output_size(28, 3, p=1) == 28, "padding 1 with a 3x3 kernel keeps the size ('same' padding)"
assert output_size(28, 2, s=2) == 14, "a 2x2 pooling with stride 2 halves the size"
assert output_size(12, 3, p=1, s=2) == 6
assert output_size(224, 7, p=3, s=2) == 112, "the first layer of a ResNet: 224 -> 112"
print("Exercise validated!")

### Exercise 3: padding and stride

Generalise Exercise 1: `conv2d(X, K, stride=1, padding=0)`.

1. Pad the image with zeros: `np.pad(X, padding)` adds `padding` rows and columns of zeros on every side.
2. The output size is `output_size(n, k, padding, stride)`.
3. The output `Y[i, j]` uses the patch that starts at row `i * stride` and column `j * stride`.

In [ ]:
def conv2d(X, K, stride=1, padding=0):
    n, k = X.shape[0], K.shape[0]
    ### START CODE HERE ###
    Y = None
    ### END CODE HERE ###
    return Y


_X = np.random.randn(7, 7)
for _s, _p in [(1, 0), (1, 1), (2, 0), (2, 1), (3, 2)]:
    _ref = F.conv2d(torch.tensor(_X).view(1, 1, 7, 7), torch.tensor(_K).view(1, 1, 3, 3), stride=_s, padding=_p).numpy()[0, 0]
    _y = conv2d(_X, _K, stride=_s, padding=_p)
    assert _y is not None and _y.shape == _ref.shape, f"stride {_s}, padding {_p}: shape {None if _y is None else _y.shape} instead of {_ref.shape}"
    assert np.allclose(_y, _ref), f"stride {_s}, padding {_p}: values differ from F.conv2d"
print("Exercise validated!")

### Exercise 4: max pooling

Write `maxpool2d(X, size=2)`: split `X` (shape `(n, n)`, `n` a multiple of `size`) into non-overlapping windows
of `size x size` and keep the maximum of each. The output has shape `(n // size, n // size)`.

The asserts use Exercise 4 of the course and compare with `F.max_pool2d(input, kernel_size)`.

In [ ]:
def maxpool2d(X, size=2):
    m = X.shape[0] // size
    ### START CODE HERE ###
    Y = None
    ### END CODE HERE ###
    return Y


_P = np.array([[1, 3, 2, 0], [4, 6, 1, 1], [0, 2, 5, 7], [1, 1, 3, 2]], dtype=float)
assert maxpool2d(_P) is not None and np.allclose(maxpool2d(_P), [[6, 2], [2, 7]]), "course Exercise 4 expects [[6, 2], [2, 7]]"
_X = np.random.randn(8, 8)
assert np.allclose(maxpool2d(_X), F.max_pool2d(torch.tensor(_X).view(1, 1, 8, 8), 2).numpy()[0, 0])
assert np.allclose(maxpool2d(_X, 4), F.max_pool2d(torch.tensor(_X).view(1, 1, 8, 8), 4).numpy()[0, 0])
print("Exercise validated!")

### Exercise 5: edge detectors on a real image

Let us apply two kernels to a FashionMNIST image at once: the vertical-edge kernel `Kv` of the course and the
horizontal-edge kernel `Kh = Kv.T`. With `F.conv2d`, several filters are simply stacked in the first
dimension of the weight: build `W` of shape `(2, 1, 3, 3)` (2 output channels, 1 input channel), then compute
`Y = F.conv2d(x, W, padding=1)` for the image `x` of shape `(1, 1, 28, 28)`.

> Hint: `torch.stack([Kv, Kh])` has shape `(2, 3, 3)`; `.unsqueeze(1)` inserts the channel dimension.

In [ ]:
fm_test = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True)
x = fm_test.data[0].float().div(255).view(1, 1, 28, 28)        # an ankle boot, pixels in [0, 1]
Kv = torch.tensor([[1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [1.0, 0.0, -1.0]])
Kh = Kv.T.contiguous()
### START CODE HERE ###
W, Y = None, None
### END CODE HERE ###

assert W is not None and W.shape == (2, 1, 3, 3), "W must have shape (C_out, C_in, k, k) = (2, 1, 3, 3)"
assert Y.shape == (1, 2, 28, 28), "with padding 1, the 28x28 size is kept, one map per filter"
assert np.allclose(Y[0, 0].numpy(), conv2d(x[0, 0].numpy(), Kv.numpy(), padding=1), atol=1e-5), "channel 0 should be the vertical-edge map"
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, im, t in zip(axes, [x[0, 0], Y[0, 0], Y[0, 1]], ["image", "vertical edges", "horizontal edges"]):
    ax.imshow(im, cmap="gray" if t == "image" else "RdBu_r"); ax.set_title(t); ax.axis("off")
plt.show()
print("Exercise validated!")

**What we see.** Each kernel lights up where the intensity changes in one direction: the sides of the boot
for the vertical kernel, the sole and the top for the horizontal one. A CNN does not use hand-made kernels:
it **learns** its kernels by gradient descent, and the first layers end up looking like edge and colour
detectors (Going further shows those of a ResNet).

# Part B: shapes and parameters

### Exercise 6: parameters of a convolution layer

A layer `nn.Conv2d(c_in, c_out, k)` has `c_out` filters; each filter spans all the input channels
(`c_in * k * k` weights) and has one bias. Write `conv_params(c_in, c_out, k)`.

In [ ]:
def conv_params(c_in, c_out, k):
    ### START CODE HERE ###
    n = None
    ### END CODE HERE ###
    return n


for _ci, _co, _k in [(1, 8, 3), (3, 64, 3), (16, 32, 5), (64, 64, 7)]:
    _layer = nn.Conv2d(_ci, _co, _k)
    assert conv_params(_ci, _co, _k) == sum(p.numel() for p in _layer.parameters()), f"wrong count for Conv2d({_ci}, {_co}, {_k})"
print("Exercise validated! A Conv2d(3, 64, 3) has", conv_params(3, 64, 3), "parameters, whatever the image size.")

### Exercise 7: predict the shapes of a network

Here is the network of Exercise 3 of the course, for 1 x 28 x 28 images. **Without running it**, fill
`predicted_shapes` with the output shape after each of the 6 modules (a tuple `(C, H, W)`, and a tuple with one
number after `nn.Flatten`), and `predicted_params` with the number of parameters of each module.

`nn.MaxPool2d(2)` has kernel 2 and stride 2. You may use `output_size` and `conv_params`.

In [ ]:
net = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1),
    nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, stride=2, padding=1),
    nn.Flatten(),
)
### START CODE HERE ###
predicted_shapes = []
predicted_params = []
### END CODE HERE ###

_h = torch.zeros(1, 1, 28, 28)
assert len(predicted_shapes) == 6 and len(predicted_params) == 6, "one shape and one count per module (6 modules)"
for _layer, _shape, _np in zip(net, predicted_shapes, predicted_params):
    _h = _layer(_h)
    assert tuple(_h.shape[1:]) == tuple(_shape), f"after {_layer}: shape {tuple(_h.shape[1:])}, you predicted {_shape}"
    assert sum(p.numel() for p in _layer.parameters()) == _np, f"{_layer}: wrong number of parameters"
print("Exercise validated!")

### Exercise 8: the receptive field

The receptive field $r$ of a layer is the size of the input region that one of its outputs sees. Going
through the layers, with $j$ the "jump" (the distance in input pixels between two neighbouring outputs):

$$r \leftarrow r + (k - 1)\, j, \qquad j \leftarrow j \times s \qquad (\text{start: } r = 1,\ j = 1)$$

Write `receptive_field(layers)` where `layers` is a list of `(k, s)` pairs (kernel size, stride), for
convolutions and poolings alike. The asserts check Exercise 5 of the course and compare with a measurement by
backpropagation (`empirical_rf`: the input pixels that receive a non-zero gradient from one output pixel).

In [ ]:
def receptive_field(layers):
    ### START CODE HERE ###
    r = None
    ### END CODE HERE ###
    return r


def empirical_rf(modules, n=64):
    x = torch.zeros(1, 1, n, n, requires_grad=True)
    y = x
    for m in modules:
        y = m(y)
    c = y.shape[-1] // 2
    y[0, 0, c, c].backward()
    rows = (x.grad[0, 0].abs().sum(dim=1) > 0).nonzero()
    return int(rows.max() - rows.min() + 1)


def ones_conv(k, s=1):
    c = nn.Conv2d(1, 1, k, stride=s, bias=False)
    nn.init.constant_(c.weight, 1.0)
    return c


assert receptive_field([(3, 1)] * 3) == 7, "three stacked 3x3 convolutions see 7x7 pixels"
assert receptive_field([(3, 1), (2, 2), (3, 1)]) == 8, "conv 3x3, pool 2, conv 3x3: 8x8"
assert receptive_field([(3, 1), (2, 2), (3, 1)]) == empirical_rf([ones_conv(3), nn.AvgPool2d(2), ones_conv(3)])
assert receptive_field([(7, 2), (3, 2), (3, 1), (3, 1)]) == empirical_rf([ones_conv(7, 2), nn.AvgPool2d(3, 2), ones_conv(3), ones_conv(3)])
print("Exercise validated! The beginning of a ResNet (conv 7/2, pool 3/2, two 3x3 convs) sees",
      receptive_field([(7, 2), (3, 2), (3, 1), (3, 1)]), "pixels.")

# Part C: a CNN versus a dense network on FashionMNIST

## Helper functions (run this cell, no need to modify it)

- The images are kept as **images**: tensors of shape `(N, 1, 28, 28)`, standardised (mean 0, std 1).
  The dense network will flatten them itself with `nn.Flatten()`.
- `train_one_epoch` and `evaluate` are the functions of sessions 4 to 6; `evaluate` returns `(loss, accuracy)`.
- `count_parameters(model)` counts the parameters.

In [ ]:
N_TRAIN_FM = 6000 if FAST_RUN else 20000
EPOCHS_FM = 3 if FAST_RUN else 5
fm_train = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True)
_idx = torch.randperm(60000, generator=torch.Generator().manual_seed(0))[:N_TRAIN_FM]
FM_MEAN, FM_STD = 0.2860, 0.3530
X_fm = ((fm_train.data[_idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)      # (N, 1, 28, 28)
y_fm = fm_train.targets[_idx]
X_fm_test = ((fm_test.data.float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)        # (10000, 1, 28, 28)
y_fm_test = fm_test.targets
fm_train_loader = DataLoader(TensorDataset(X_fm, y_fm), batch_size=64, shuffle=True)
fm_test_loader = DataLoader(TensorDataset(X_fm_test, y_fm_test), batch_size=1000)
loss_fn = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / n


def evaluate(model, loader, loss_fn):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total += loss_fn(out, yb).item() * len(yb)
            correct += (out.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total / n, correct / n


def count_parameters(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


print("FashionMNIST train", tuple(X_fm.shape), "test", tuple(X_fm_test.shape))

### Exercise 9: the dense baseline

Write `make_dense()`: `nn.Flatten()` (turns `(N, 1, 28, 28)` into `(N, 784)`), then the network of session 5,
`784 -> 256 -> 128 -> 10` with ReLU.

In [ ]:
def make_dense():
    ### START CODE HERE ###
    model = None
    ### END CODE HERE ###
    return model


_m = make_dense()
assert _m is not None and _m(torch.zeros(5, 1, 28, 28)).shape == (5, 10)
assert count_parameters(_m) == 235_146, f"expected 235 146 parameters, found {count_parameters(_m)}"
print("Exercise validated!")

### Exercise 10: a small CNN as an `nn.Module` class

Complete the class `SmallCNN` (the diagram of the course):

| Step | Module | Output shape |
|---|---|---|
| input | | 1 x 28 x 28 |
| `self.conv1 = nn.Conv2d(1, 16, 3, padding=1)`, then ReLU | conv | 16 x 28 x 28 |
| `self.pool = nn.MaxPool2d(2)` | pool | 16 x 14 x 14 |
| `self.conv2 = nn.Conv2d(16, 32, 3, padding=1)`, then ReLU | conv | 32 x 14 x 14 |
| `self.pool` again (a pooling layer has no parameter, it can be reused) | pool | 32 x 7 x 7 |
| `x.flatten(1)` (flatten everything except the batch dimension) | | 1568 |
| `self.fc = nn.Linear(32 * 7 * 7, 10)` | linear | 10 |

`__init__` declares the layers (after `super().__init__()`, already written); `forward` uses them. `torch.relu(t)`
applies ReLU to a tensor.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        ### START CODE HERE ###

        ### END CODE HERE ###

    def forward(self, x):
        ### START CODE HERE ###
        out = None
        ### END CODE HERE ###
        return out


_m = SmallCNN()
_o = _m(torch.zeros(5, 1, 28, 28))
assert _o is not None and _o.shape == (5, 10), "the output must have shape (batch, 10)"
assert count_parameters(_m) == 20_490, f"expected 20 490 parameters, found {count_parameters(_m)}"
print(f"Exercise validated! SmallCNN: {count_parameters(_m)} parameters, "
      f"{count_parameters(make_dense()) / count_parameters(_m):.1f} times fewer than the dense network.")

### Exercise 11: train both and compare

For each of the two models (`make_dense()` and `SmallCNN()`, created after `torch.manual_seed(0)`), create an
optimiser `torch.optim.Adam(model.parameters(), lr=1e-3)`, train for `EPOCHS_FM` epochs with `train_one_epoch`
on `fm_train_loader`, then compute the test accuracy with `evaluate` on `fm_test_loader`. Store the models in
`dense` and `cnn` and the accuracies in `acc_dense` and `acc_cnn`.

> Hint: write a small function `train(model)` that does the loop and returns the test accuracy, and call it twice.

In [ ]:
t0 = time.time()
### START CODE HERE ###
acc_dense, acc_cnn = None, None
### END CODE HERE ###

assert acc_dense is not None and acc_cnn is not None, "compute both accuracies"
assert acc_dense > 0.75, f"the dense network should reach 0.75, found {acc_dense:.3f}"
assert acc_cnn > acc_dense, f"the CNN should beat the dense network ({acc_cnn:.3f} versus {acc_dense:.3f})"
print(f"Exercise validated! Dense: {acc_dense:.3f} with {count_parameters(dense)} parameters. "
      f"CNN: {acc_cnn:.3f} with {count_parameters(cnn)} parameters. ({time.time() - t0:.0f} s)")

### Exercise 12: shift the test images

A dense network has one weight per pixel position: if the object moves, other weights are used. A CNN slides
the same filters everywhere. Let us test it: shift every test image 3 pixels to the right with
`torch.roll(X, shifts=3, dims=3)` (dimension 3 is the width; the columns that leave on the right come back on
the left, which is only background here), then measure both test accuracies again.

Store `acc_dense_shift` and `acc_cnn_shift`.

> Hint: build a new `DataLoader(TensorDataset(X_shift, y_fm_test), batch_size=1000)` and reuse `evaluate`.

In [ ]:
### START CODE HERE ###
acc_dense_shift, acc_cnn_shift = None, None
### END CODE HERE ###

assert acc_dense_shift is not None and acc_dense_shift < acc_dense - 0.2, "the dense network should suffer a lot"
assert acc_cnn_shift > acc_dense_shift, "the CNN should resist better"
print(f"Exercise validated! Shifted by 3 pixels: dense {acc_dense:.3f} -> {acc_dense_shift:.3f}, "
      f"CNN {acc_cnn:.3f} -> {acc_cnn_shift:.3f}")

**What we see.** The CNN is more accurate with about 11 times fewer parameters: locality (a filter looks at a
small neighbourhood) and weight sharing (the same filter everywhere) are the right assumptions for images.
Shifting the images hurts both, but the dense network much more. Our CNN is not fully shift-invariant (its
last layer is dense, and FashionMNIST objects are always centred during training): data augmentation with
random shifts would fix the rest.

# Part D: transfer learning with a pretrained ResNet-18

## Helper functions (run this cell, no need to modify it)

- A subset of **CIFAR-10** (32 x 32 colour images, 10 classes: plane, car, bird, cat, deer, dog, frog, horse,
  ship, truck).
- `prepare(X)`: the ResNet was trained on ImageNet photos of 224 x 224 pixels, normalised with the ImageNet
  mean and standard deviation of each colour channel. We upsample our images to 64 x 64 (a compromise: 224 would
  be more accurate but too slow on a CPU) with `F.interpolate`, then normalise with the ImageNet statistics.
  **A pretrained model must always receive inputs prepared like during its pre-training.**
- `fit_transfer(model, optimizer, n_epochs)`: trains and returns the test accuracy after each epoch. It keeps
  the model in `eval()` mode during training: with a few hundred images we do not want to change the batch-norm
  running statistics learned on ImageNet (the weights are still trained normally).

In [ ]:
N_TR_CIFAR = 500 if FAST_RUN else 2000
N_TE_CIFAR = 500 if FAST_RUN else 1000
EPOCHS_HEAD = 2 if FAST_RUN else 3
EPOCHS_FT = 2 if FAST_RUN else 3
IMG_SIZE = 64
CIFAR_CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
c_train = datasets.CIFAR10(root=DATA_DIR, train=True, download=True)
c_test = datasets.CIFAR10(root=DATA_DIR, train=False, download=True)
_g = torch.Generator().manual_seed(0)
_itr = torch.randperm(50000, generator=_g)[:N_TR_CIFAR].numpy()
_ite = torch.randperm(10000, generator=_g)[:N_TE_CIFAR].numpy()
Xc_train = torch.tensor(c_train.data[_itr]).permute(0, 3, 1, 2).float() / 255     # (N, 3, 32, 32), in [0, 1]
yc_train = torch.tensor(np.array(c_train.targets)[_itr])
Xc_test = torch.tensor(c_test.data[_ite]).permute(0, 3, 1, 2).float() / 255
yc_test = torch.tensor(np.array(c_test.targets)[_ite])
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def prepare(X):
    X = F.interpolate(X, size=IMG_SIZE, mode="bilinear", align_corners=False)
    return (X - IMAGENET_MEAN) / IMAGENET_STD


c_train_loader = DataLoader(TensorDataset(prepare(Xc_train), yc_train), batch_size=64, shuffle=True)
c_test_loader = DataLoader(TensorDataset(prepare(Xc_test), yc_test), batch_size=250)


def fit_transfer(model, optimizer, n_epochs):
    torch.manual_seed(1)
    accs = []
    for epoch in range(n_epochs):
        model.eval()                                   # batch norm keeps its ImageNet statistics
        for xb, yb in c_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        accs.append(evaluate(model, c_test_loader, loss_fn)[1])
        print(f"epoch {epoch + 1}: test accuracy {accs[-1]:.3f}")
    return accs


print("CIFAR-10 subset: train", tuple(Xc_train.shape), "test", tuple(Xc_test.shape))
fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for k, ax in enumerate(axes):
    ax.imshow(Xc_train[k].permute(1, 2, 0)); ax.set_title(CIFAR_CLASSES[yc_train[k]], fontsize=10); ax.axis("off")
plt.show()

### Exercise 13: load the model and replace its head

`resnet18(weights=ResNet18_Weights.DEFAULT)` builds a ResNet-18 and loads the weights learned on ImageNet
(1.2 million images, 1000 classes; downloaded once, 45 MB). Its last layer `model_tl.fc` is an
`nn.Linear(512, 1000)`: 1000 ImageNet classes. Replace it by a new `nn.Linear(512, 10)` for our 10 classes
(a simple assignment: `model_tl.fc = ...`). The new layer starts with random weights.

In [ ]:
torch.manual_seed(0)
model_tl = resnet18(weights=ResNet18_Weights.DEFAULT)
print("original head:", model_tl.fc)
### START CODE HERE ###

### END CODE HERE ###
model_tl = model_tl.to(device)

assert model_tl.fc.out_features == 10, "the new head must output 10 scores"
assert count_parameters(model_tl) == 11_181_642, f"expected 11 181 642 parameters, found {count_parameters(model_tl)}"
print("Exercise validated! New head:", model_tl.fc)

### Exercise 14: freeze the backbone

We first train only the new head. To freeze a parameter, set `p.requires_grad = False`: autograd no longer
computes its gradient and the optimiser will not touch it.

1. Freeze all the parameters of `model_tl` (loop over `model_tl.parameters()`).
2. Unfreeze those of the head (`model_tl.fc.parameters()`).

Then compute `n_trainable` with `count_parameters(model_tl, trainable_only=True)`.

In [ ]:
### START CODE HERE ###
n_trainable = None
### END CODE HERE ###

assert n_trainable == 5130, f"only the head should be trainable (512 x 10 + 10 = 5130), found {n_trainable}"
assert not model_tl.conv1.weight.requires_grad, "the first convolution must be frozen"
print("Exercise validated! Trainable:", n_trainable, "of", count_parameters(model_tl))

### Exercise 15: train the head

Create an Adam optimiser on the head only (`model_tl.fc.parameters()`, `lr=3e-3`: a new layer can learn fast) and train with
`accs_head = fit_transfer(model_tl, optimizer, EPOCHS_HEAD)`. Store the final test accuracy in `acc_head`.

In [ ]:
t0 = time.time()
### START CODE HERE ###
acc_head = None
### END CODE HERE ###

assert acc_head is not None and acc_head > 0.40, f"the head alone should reach 0.40, found {acc_head}"
print(f"Exercise validated! Head only: {acc_head:.3f} after {EPOCHS_HEAD} epochs ({time.time() - t0:.0f} s). Chance level is 0.10.")

### Exercise 16: fine-tune the last block

The last block of the ResNet (`model_tl.layer4`, 8.4 million parameters) computes the most specific features,
the ones most tied to ImageNet. Unfreeze it and train it together with the head, with a **smaller learning
rate** for the pretrained weights (we only want to adjust them, not destroy them). An optimiser accepts
**parameter groups**, each with its own learning rate:

```python
torch.optim.Adam([{"params": model_tl.layer4.parameters(), "lr": 1e-4},
                  {"params": model_tl.fc.parameters(), "lr": 1e-3}])
```

Unfreeze `layer4`, create this optimiser, train for `EPOCHS_FT` epochs and store the final accuracy in `acc_ft`.

In [ ]:
t0 = time.time()
### START CODE HERE ###
acc_ft = None
### END CODE HERE ###

assert count_parameters(model_tl, trainable_only=True) == 8_398_858, "layer4 and fc should be trainable"
assert acc_ft is not None and acc_ft > acc_head + 0.02, f"fine-tuning should improve on the head alone ({acc_ft} versus {acc_head:.3f})"
print(f"Exercise validated! Fine-tuned: {acc_ft:.3f} (head only {acc_head:.3f}) in {time.time() - t0:.0f} s.")

**What we see.** With a few hundred or a few thousand images, the pretrained ResNet is far above what a network
trained from scratch could reach: its filters, learned on 1.2 million photos, already detect edges, textures and
object parts. Training only the head is fast and safe; fine-tuning the last block adapts the most specific
features to the new task. With more data (or more similar to ImageNet), fine-tune more layers. This is exactly
what you will do on the artworks of the project.

# Going further

### Exercise 17 (bonus): what the first layer has learned

`model_tl.conv1` is the first convolution of the ResNet: 64 filters of size 3 x 7 x 7 (one 7 x 7 kernel per
colour channel). Store its weight tensor (detached from the graph with `.detach().cpu()`) in `filters`, and
compute `maps`, the 64 feature maps that `conv1` produces for the first prepared training image
(`prepare(Xc_train[:1])`, shape `(1, 3, 64, 64)`). The cell plots them.

> Hint: `conv1` has stride 2 and padding 3: what is the output size for a 64 x 64 image?

In [ ]:
### START CODE HERE ###
filters, maps = None, None
### END CODE HERE ###

assert filters is not None and filters.shape == (64, 3, 7, 7), "conv1 has 64 filters of shape 3 x 7 x 7"
assert maps is not None and maps.shape == (1, 64, output_size(64, 7, 3, 2), output_size(64, 7, 3, 2)), "64 maps of 32 x 32"
fig, axes = plt.subplots(4, 16, figsize=(12, 3.2))
for k, ax in enumerate(axes.flat):
    w = filters[k].permute(1, 2, 0)
    ax.imshow((w - w.min()) / (w.max() - w.min())); ax.axis("off")
plt.suptitle("the 64 first-layer filters of ResNet-18")
plt.show()
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for k, ax in enumerate(axes.flat):
    ax.imshow(maps[0, k], cmap="viridis"); ax.axis("off")
plt.suptitle("16 of the 64 feature maps for one image")
plt.show()
print("Exercise validated!")